In [1]:
# Chunk: Test model on two custom scenarios (normal vs drop)
import pandas as pd
import joblib
import numpy as np

# 1. Load trained model
model_path = "../Models/tuned_xgb.pkl"   # or baseline_xgb.pkl
model = joblib.load(model_path)
print(f"Model loaded from {model_path}\n")

# 2. Load feature order from training data (critical!)
X_train_sample = pd.read_csv("../Data/Splits/X_train_smote.csv")
feature_columns = X_train_sample.columns.tolist()

# 3. Helper function to predict from a dictionary of features
def predict_from_dict(input_dict):
    # Fill missing features with average values from training (for simplicity)
    # In real use, you should provide all 27 features.
    avg_values = X_train_sample.mean().to_dict()
    for col in feature_columns:
        if col not in input_dict:
            input_dict[col] = avg_values[col]
    # Create DataFrame with correct column order
    X_input = pd.DataFrame([input_dict], columns=feature_columns, dtype=float)
    prob = model.predict_proba(X_input)[0][1]
    pred = model.predict(X_input)[0]
    return pred, prob

# 4. Define a NORMAL call (should output 0)
normal_input = {
    'rsrp_min': -82,
    'rsrp_max': -72,
    'rsrp_mean': -77,
    'rsrp_std': 1.5,
    'rsrp_last': -80,
    'rsrp_slope_last5': 0.05,
    'rsrp_slope_all': -0.02,
    'rsrp_time_below_minus110': 0,
    'rsrq_min': -12,
    'rsrq_mean': -10,
    'rsrq_std': 0.8,
    'rsrq_slope_last5': 0.0,
    'sinr_min': 12,
    'sinr_mean': 18,
    'sinr_time_below_0': 0,
    'rain_mean': 0.0,
    'rain_max': 0.0,
    'tower_load_mean': 40,
    'tower_load_max': 45,
    'tower_load_high': 0,
    'speed_kmph_mean': 3.0,
    'speed_kmph_max': 3.5,
    'call_duration_sec': 120,
    'start_hour': 10,
    'band_5G_n78': 0,
    'band_LTE_B20': 1,
    'band_LTE_B3': 0
}
pred_norm, prob_norm = predict_from_dict(normal_input)
print("=== NORMAL SCENARIO ===")
print(f"Prediction: {pred_norm} (0 = normal, 1 = drop)")
print(f"Drop probability: {prob_norm:.4f}\n")

# 5. Define a DROP call (should output 1)
drop_input = {
    'rsrp_min': -118,
    'rsrp_max': -85,
    'rsrp_mean': -102,
    'rsrp_std': 6.5,
    'rsrp_last': -118,
    'rsrp_slope_last5': -3.5,
    'rsrp_slope_all': -2.2,
    'rsrp_time_below_minus110': 12,
    'rsrq_min': -19,
    'rsrq_mean': -16,
    'rsrq_std': 2.5,
    'rsrq_slope_last5': -1.8,
    'sinr_min': -4,
    'sinr_mean': 0,
    'sinr_time_below_0': 8,
    'rain_mean': 8.0,
    'rain_max': 15.0,
    'tower_load_mean': 88,
    'tower_load_max': 95,
    'tower_load_high': 25,
    'speed_kmph_mean': 85.0,
    'speed_kmph_max': 90.0,
    'call_duration_sec': 35,
    'start_hour': 18,
    'band_5G_n78': 1,
    'band_LTE_B20': 0,
    'band_LTE_B3': 0
}
pred_drop, prob_drop = predict_from_dict(drop_input)
print("=== DROP SCENARIO ===")
print(f"Prediction: {pred_drop} (0 = normal, 1 = drop)")
print(f"Drop probability: {prob_drop:.4f}")

Model loaded from ../Models/tuned_xgb.pkl

=== NORMAL SCENARIO ===
Prediction: 0 (0 = normal, 1 = drop)
Drop probability: 0.0030

=== DROP SCENARIO ===
Prediction: 1 (0 = normal, 1 = drop)
Drop probability: 0.9969


In [2]:
import pandas as pd
import joblib

# ------------------------------------------------------------------
# 1. Load the trained model (choose 'baseline_xgb.pkl' or 'tuned_xgb.pkl')
# ------------------------------------------------------------------
model_path = "../Models/tuned_xgb.pkl"   # change to baseline if you prefer
try:
    model = joblib.load(model_path)
    print(f"Model loaded from {model_path}")
except FileNotFoundError:
    print(f"Model not found at {model_path}. Please train and save first.")
    exit(1)

# ------------------------------------------------------------------
# 2. Define the exact feature columns (order matters!)
#    (copy from your engineered dataset columns, excluding 'is_drop')
# ------------------------------------------------------------------
feature_columns = [
    'rsrp_min', 'rsrp_max', 'rsrp_mean', 'rsrp_std', 'rsrp_last',
    'rsrp_slope_last5', 'rsrp_slope_all', 'rsrp_time_below_minus110',
    'rsrq_min', 'rsrq_mean', 'rsrq_std', 'rsrq_slope_last5',
    'sinr_min', 'sinr_mean', 'sinr_time_below_0',
    'rain_mean', 'rain_max',
    'tower_load_mean', 'tower_load_max', 'tower_load_high',
    'speed_kmph_mean', 'speed_kmph_max',
    'call_duration_sec', 'start_hour',
    'band_5G_n78', 'band_LTE_B20', 'band_LTE_B3'
]

# ------------------------------------------------------------------
# 3. Define helper function to predict a single row
# ------------------------------------------------------------------
def predict_drop(feature_values):
    """
    feature_values: list or dict with values for all features (in same order as feature_columns)
    Returns: 1 if drop predicted, 0 otherwise.
    """
    # Convert to DataFrame with proper column names
    if isinstance(feature_values, dict):
        df = pd.DataFrame([feature_values])
        # Ensure columns are in the correct order
        df = df[feature_columns]
    else:
        # assume list in correct order
        df = pd.DataFrame([feature_values], columns=feature_columns)

    prob = model.predict_proba(df)[0][1]   # probability of drop
    pred = model.predict(df)[0]
    print(f"Prediction probability (drop) = {prob:.4f}")
    return pred

# ------------------------------------------------------------------
# 4. Test Case 1: A call that is very likely to drop
# ------------------------------------------------------------------
print("\n" + "="*50)
print("TEST CASE 1: DROP SCENARIO")
drop_values = {
    'rsrp_min': -118,
    'rsrp_max': -85,
    'rsrp_mean': -105,
    'rsrp_std': 5.5,
    'rsrp_last': -118,
    'rsrp_slope_last5': -3.2,
    'rsrp_slope_all': -2.1,
    'rsrp_time_below_minus110': 8,
    'rsrq_min': -18,
    'rsrq_mean': -15,
    'rsrq_std': 2.0,
    'rsrq_slope_last5': -1.5,
    'sinr_min': -3,
    'sinr_mean': 2,
    'sinr_time_below_0': 5,
    'rain_mean': 5.0,
    'rain_max': 12.0,
    'tower_load_mean': 85,
    'tower_load_max': 92,
    'tower_load_high': 15,       # seconds above 80% load
    'speed_kmph_mean': 80.0,
    'speed_kmph_max': 85.0,
    'call_duration_sec': 45,
    'start_hour': 18,            # peak hour
    'band_5G_n78': 1,
    'band_LTE_B20': 0,
    'band_LTE_B3': 0
}
pred_drop = predict_drop(drop_values)
print(f"PREDICTION: {pred_drop}  (1 = drop, 0 = normal)")

# ------------------------------------------------------------------
# 5. Test Case 2: A normal call (should not drop)
# ------------------------------------------------------------------
print("\n" + "="*50)
print("TEST CASE 2: NORMAL SCENARIO")
normal_values = {
    'rsrp_min': -85,
    'rsrp_max': -75,
    'rsrp_mean': -80,
    'rsrp_std': 1.5,
    'rsrp_last': -82,
    'rsrp_slope_last5': 0.1,
    'rsrp_slope_all': -0.05,
    'rsrp_time_below_minus110': 0,
    'rsrq_min': -12,
    'rsrq_mean': -10,
    'rsrq_std': 0.8,
    'rsrq_slope_last5': 0.0,
    'sinr_min': 12,
    'sinr_mean': 18,
    'sinr_time_below_0': 0,
    'rain_mean': 0.0,
    'rain_max': 0.0,
    'tower_load_mean': 40,
    'tower_load_max': 45,
    'tower_load_high': 0,
    'speed_kmph_mean': 3.0,
    'speed_kmph_max': 4.0,
    'call_duration_sec': 120,
    'start_hour': 10,            # off‑peak
    'band_5G_n78': 0,
    'band_LTE_B20': 1,
    'band_LTE_B3': 0
}
pred_normal = predict_drop(normal_values)
print(f"PREDICTION: {pred_normal}  (1 = drop, 0 = normal)")

# ------------------------------------------------------------------
# 6. Optional: interactive test (uncomment if you want to input values)
# ------------------------------------------------------------------
# print("\n" + "="*50)
# print("INTERACTIVE TEST (enter custom values)")
# user_dict = {}
# for col in feature_columns:
#     val = input(f"Enter {col}: ")
#     try:
#         user_dict[col] = float(val)
#     except:
#         user_dict[col] = 0.0
# pred_user = predict_drop(user_dict)
# print(f"Prediction: {pred_user}")

Model loaded from ../Models/tuned_xgb.pkl

TEST CASE 1: DROP SCENARIO
Prediction probability (drop) = 0.9957
PREDICTION: 1  (1 = drop, 0 = normal)

TEST CASE 2: NORMAL SCENARIO
Prediction probability (drop) = 0.0036
PREDICTION: 0  (1 = drop, 0 = normal)
